# 13. GitHub Actions as a CI Demo

The previous steps made the project executable: we refactored notebook code into Python modules, added tests, exposed the model through FastAPI, added a Streamlit frontend, and packaged the application with Docker.


GitHub Actions now demonstrates a simple CI workflow. CI means continuous integration: every relevant code change is checked automatically in a clean environment.

## 1. What This Workflow Does and Does Not Do

This workflow is intentionally limited, but it already covers more than simple unit testing.

It does:

- install the project dependencies
- run the test suite with `pytest`
- build the Docker images
- start the API container
- call the `/health` endpoint as a smoke test
- push the API image to GitHub Container Registry on `main`

It does not:

- deploy to a running cloud service
- expose a public production API URL
- run continuous training
- require production secrets
- manage model registry stages or rollout approvals

This keeps the first CI/CD step focused. The workflow verifies code behavior, checks whether the container can be built and started, and publishes a versioned API image as a deployable artifact on `main`. The image is available for deployment, but the workflow does not yet start it in a cloud runtime.


## 2. Workflow File

GitHub Actions workflows live in `.github/workflows/`.

For this project, the CI workflow is stored here:

```text
.github/workflows/ci.yml
```

The workflow runs on pull requests to `main` and on pushes to `main`.

## 3. Test Job

The first job checks whether the Python project still behaves as expected.

Conceptually, it performs these steps:

```bash
python -m pip install --upgrade pip
pip install -r requirements-api.txt
pip install --no-deps xgboost
pip install pytest httpx
pytest
```

The workflow installs a focused test environment instead of the full course `requirements.txt`. The full file also contains notebook, documentation, and optional exercise dependencies. CI should install what it needs for this check, not every package used anywhere in the course repository.

The difference to a local run is that GitHub Actions runs the check in a fresh environment. That makes hidden local assumptions more visible.

## 4. Docker Build Job

Passing tests does not automatically mean the application can be packaged and started as a container.

The second job therefore builds the Docker images:

```bash
docker compose build
```

This checks whether the Dockerfiles, dependency files, and copied application paths are still consistent.

## 5. API Smoke Test

After the image is built, the workflow starts only the API container:

```bash
docker compose up -d api
```

Then it calls the health endpoint:

```bash
curl --fail http://127.0.0.1:8000/health
```

This is a smoke test. It does not prove that every detail of the service works, but it verifies that the container starts and the API responds.

The workflow does not call `/predict`, because the trained model artifacts are intentionally not committed to Git. A full prediction test would require a controlled test artifact, an artifact download step, or a registry-backed model loading mechanism.

## 6. CI, Delivery, and Deployment

It is useful to separate three ideas that are often grouped under CI/CD.

| Step | What happens in this project? | Result |
|---|---|---|
| Continuous Integration | Tests run, Docker images build, API health check passes | The change is technically checked |
| Container Delivery | On `main`, the API image is pushed to GitHub Container Registry | A versioned deployable artifact exists |
| Deployment | A cloud service pulls the image and starts it as a running API | Not implemented in this project |

The GitHub Container Registry push is therefore not just a test. It creates a deployable container image. A cloud platform could later pull this image and run it, for example in Azure Container Apps, Azure App Service for Containers, Kubernetes, AWS ECS, or Google Cloud Run.

The important boundary is this:

```text
Build and push image -> deployable artifact exists
Deploy image          -> service is running in a target environment
```

This notebook implements the first two parts. It does not implement the final cloud deployment step.


## 7. Publishing the API Image

After a successful smoke test, pushes to `main` publish the API image to GitHub Container Registry.

The image name follows this pattern:

```text
ghcr.io/<owner>/<repo>/bank-marketing-api:<tag>
```

The workflow uses two tags:

- the commit SHA, for example `8f3a...`, which is reproducible
- `latest`, only for the current state of `main`

Pull requests do not push images. They only build and test. This avoids publishing container artifacts for every experimental branch.

The workflow uses GitHub's built-in `GITHUB_TOKEN`. For this repository-local package push, no separate secret is needed. The workflow only needs package write permission:

```yaml
permissions:
  contents: read
  packages: write
```

At this point, the API image is available in the registry and could be consumed by a deployment workflow. For example, a later Azure deployment could select the commit-SHA tag, pull the image from GHCR, and run it as a container app.

The pushed image still does not contain the trained model artifacts. It contains the serving code and runtime environment. The model artifacts are a separate lifecycle and would be provided by a model registry, object storage, or a controlled manual upload in a later deployment setup.


## 8. From CI to CD

This workflow still stops before deployment.

A later professional deployment workflow could add these steps:

- select one image tag from GitHub Container Registry
- deploy the selected image to a cloud service
- load model artifacts from a model registry or object storage

The important distinction is that CI verifies the project and produces confidence. CD uses versioned artifacts and deploys them into a target environment.

## 9. Reflection

The CI workflow is deliberately small, but it already captures an important MLOps idea: the system should be checked automatically before changes are merged or released.

For ML applications, this should include more than unit tests. The application must also be packageable, startable, and reachable through its serving interface.

In this project, GitHub Actions checks exactly that first layer:

- code-level tests pass
- Docker images build
- the API container starts
- `/health` responds
- the API image is pushed to GHCR on `main`

This means the project produces a versioned delivery artifact. It is ready to be used by a later deployment workflow, but it is not yet deployed as a running cloud service.

The next level would be to add a deployment target. A cloud runtime would pull one selected image tag from the registry, inject the required configuration, connect model artifacts, and expose the API through a managed endpoint.
